# Credit Fraud — Training & MLflow Tracking

Interactive mirror of `train.py`. Each cell maps to a logical step in the pipeline;
every model configuration is logged as a separate run under the `credit-fraud` MLflow experiment.
Run `mlflow ui` in the project root to browse results after training.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import plotly.express as px
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from features import build_features, BASELINE_COLS, ENGINEERED_COLS

EXPERIMENT = "credit-fraud"
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
TARGET_COL = "target"    # TODO: update to actual target column name
ID_COLS: list[str] = []  # TODO: update, e.g. ["id", "loan_id"]

mlflow.set_experiment(EXPERIMENT)
print(f"MLflow experiment: {EXPERIMENT}")

## Load Data

In [ ]:
train = pd.read_csv("data/train.csv")
y = train[TARGET_COL].values
drop_cols = [TARGET_COL] + ID_COLS
print(f"Train shape: {train.shape}  |  Positive rate: {y.mean():.2%}  |  Imbalance ~1:{int((1-y.mean())/y.mean()+0.5)}")
train.head(3)

## Feature Engineering

Calls `build_features` (defined in `features.py`) to impute, encode, and optionally engineer features.
Produces up to two feature sets — `baseline` (raw) and `engineered` (+ transforms / interactions) —
so both are evaluated in the grid search.

In [ ]:
X_all, feat_cols, enc, scl = build_features(
    train.drop(columns=drop_cols, errors="ignore")
)
print(f"Feature matrix: {X_all.shape}")

feat_sets: dict[str, pd.DataFrame] = {}
if BASELINE_COLS:
    feat_sets["baseline"] = X_all[BASELINE_COLS]
if ENGINEERED_COLS:
    feat_sets["engineered"] = X_all[ENGINEERED_COLS]
if not feat_sets:
    feat_sets["all"] = X_all
    print(
        f"NOTE: BASELINE_COLS and ENGINEERED_COLS are empty. "
        f"Training on all {X_all.shape[1]} numeric features.\n"
        f"Fill in features.py after running EDA, then re-run."
    )

for name, X_fs in feat_sets.items():
    print(f"  {name}: {X_fs.shape[1]} features")

## Cross-Validation Utilities

`cv_metrics` runs 5-fold stratified CV and sweeps thresholds 0.10–0.90 per fold to find
the one that maximises F1. `run_experiment` wraps a single config in an MLflow run.

In [ ]:
def cv_metrics(model, X, y) -> dict:
    auc_scores, pr_auc_scores, f1_default = [], [], []
    f1_tuned, prec_tuned, rec_tuned, best_thresholds = [], [], [], []

    for tr_idx, val_idx in CV.split(X, y):
        X_tr = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_val = X.iloc[val_idx] if hasattr(X, "iloc") else X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        m = clone(model)
        m.fit(X_tr, y_tr)
        proba = m.predict_proba(X_val)[:, 1]

        auc_scores.append(roc_auc_score(y_val, proba))
        pr_auc_scores.append(average_precision_score(y_val, proba))
        f1_default.append(f1_score(y_val, (proba >= 0.5).astype(int), zero_division=0))

        best_t, best_f = 0.5, 0.0
        for t in np.arange(0.10, 0.91, 0.02):
            f = f1_score(y_val, (proba >= t).astype(int), zero_division=0)
            if f > best_f:
                best_f, best_t = f, float(t)

        preds_t = (proba >= best_t).astype(int)
        f1_tuned.append(best_f)
        best_thresholds.append(best_t)
        prec_tuned.append(precision_score(y_val, preds_t, zero_division=0))
        rec_tuned.append(recall_score(y_val, preds_t, zero_division=0))

    return {
        "cv_auc":             float(np.mean(auc_scores)),
        "cv_pr_auc":          float(np.mean(pr_auc_scores)),
        "cv_f1":              float(np.mean(f1_default)),
        "cv_f1_tuned":        float(np.mean(f1_tuned)),
        "cv_precision_tuned": float(np.mean(prec_tuned)),
        "cv_recall_tuned":    float(np.mean(rec_tuned)),
        "cv_threshold":       float(np.mean(best_thresholds)),
    }


def run_experiment(name: str, model, params: dict, X, y, feature_set: str):
    with mlflow.start_run(run_name=f"{name}_{feature_set}"):
        mlflow.set_tag("model", name.split("_")[0])
        mlflow.set_tag("feature_set", feature_set)
        mlflow.set_tag("threshold_strategy", "tuned")
        mlflow.log_params({k: str(v) for k, v in params.items()})

        metrics = cv_metrics(model, X, y)
        mlflow.log_metrics(metrics)
        mlflow.log_param("optimal_threshold", metrics["cv_threshold"])

        model.fit(X, y)
        mlflow.sklearn.log_model(model, "model")

    return metrics["cv_pr_auc"], model, metrics


print("cv_metrics and run_experiment defined.")

## Model Grid

Builds the full hyperparameter grid: LogReg × 4 C values, RF × 4, GBM × 8, XGBoost × 8, LightGBM × 8.
Each entry is `(display_name, estimator, logged_params_dict)`.

In [ ]:
def make_model_grid():
    grid = []

    for C in [0.01, 0.1, 1.0, 10.0]:
        grid.append((
            f"logreg_C{C}",
            make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000, solver="lbfgs")),
            {"C": C, "solver": "lbfgs", "max_iter": 1000},
        ))

    for n_est in [100, 300]:
        for max_d in [None, 10]:
            grid.append((
                f"rf_n{n_est}_d{max_d or 'None'}",
                RandomForestClassifier(n_estimators=n_est, max_depth=max_d, random_state=42, n_jobs=-1),
                {"n_estimators": n_est, "max_depth": max_d},
            ))

    for n_est in [100, 200]:
        for max_d in [3, 5]:
            for lr in [0.05, 0.1]:
                grid.append((
                    f"gb_n{n_est}_d{max_d}_lr{lr}",
                    GradientBoostingClassifier(
                        n_estimators=n_est, max_depth=max_d, learning_rate=lr, random_state=42
                    ),
                    {"n_estimators": n_est, "max_depth": max_d, "learning_rate": lr},
                ))

    for n_est in [100, 300]:
        for max_d in [3, 6]:
            for lr in [0.05, 0.1]:
                grid.append((
                    f"xgb_n{n_est}_d{max_d}_lr{lr}",
                    XGBClassifier(
                        n_estimators=n_est, max_depth=max_d, learning_rate=lr,
                        eval_metric="logloss", random_state=42, verbosity=0,
                    ),
                    {"n_estimators": n_est, "max_depth": max_d, "learning_rate": lr},
                ))

    for n_est in [100, 300]:
        for num_leaves in [31, 63]:
            for lr in [0.05, 0.1]:
                grid.append((
                    f"lgbm_n{n_est}_l{num_leaves}_lr{lr}",
                    LGBMClassifier(
                        n_estimators=n_est, num_leaves=num_leaves, learning_rate=lr,
                        random_state=42, verbose=-1,
                    ),
                    {"n_estimators": n_est, "num_leaves": num_leaves, "learning_rate": lr},
                ))

    return grid


grid = make_model_grid()
total_runs = len(grid) * len(feat_sets)
print(f"Grid: {len(grid)} configs × {len(feat_sets)} feature set(s) = {total_runs} MLflow runs")

## Train All Models

Runs the full grid. Each config is evaluated with 5-fold stratified CV, then fit on the full
training set and logged to MLflow. **This cell can take several minutes to complete.**

In [ ]:
# results[key] = (cv_pr_auc, fitted_model, base_model_name, feature_set_name, metrics_dict)
results: dict[str, tuple] = {}

for feat_name, X_feat in feat_sets.items():
    print(f"\n=== Feature set: {feat_name} ({X_feat.shape[1]} features) ===")
    for name, model, params in grid:
        pr_auc, fitted, metrics = run_experiment(name, model, params, X_feat, y, feat_name)
        results[f"{name}__{feat_name}"] = (pr_auc, fitted, name.split("_")[0], feat_name, metrics)
        print(
            f"  {name:38s} [{feat_name:12s}]"
            f"  auc={metrics['cv_auc']:.4f}"
            f"  pr_auc={metrics['cv_pr_auc']:.4f}"
            f"  f1_tuned={metrics['cv_f1_tuned']:.4f}"
            f"  @t={metrics['cv_threshold']:.2f}"
        )

print("\nDone.")

## Results Summary

In [ ]:
rows = []
for key, (pr_auc, _, model_name, feat_name, metrics) in results.items():
    rows.append({
        "run_key":     key,
        "model":       model_name,
        "feature_set": feat_name,
        **{k: round(v, 4) for k, v in metrics.items()},
    })

results_df = (
    pd.DataFrame(rows)
    .sort_values("cv_pr_auc", ascending=False)
    .reset_index(drop=True)
)
results_df.head(10)

In [ ]:
fig = px.bar(
    results_df.head(20),
    x="run_key",
    y="cv_pr_auc",
    color="model",
    title="Top-20 Runs by CV PR-AUC",
    labels={"run_key": "Run", "cv_pr_auc": "CV PR-AUC"},
)
fig.update_layout(xaxis_tickangle=-45, showlegend=True)
fig.show()

## Voting Ensemble (Top-3)

Soft-voting ensemble from the three best models on the highest-priority feature set.
Uses `clone()` on the already-fitted models to create fresh copies for re-fitting inside CV.

In [ ]:
best_feat = "engineered" if "engineered" in feat_sets else list(feat_sets.keys())[-1]
X_best = feat_sets[best_feat]

top3 = sorted(
    [(k, v) for k, v in results.items() if v[3] == best_feat],
    key=lambda x: x[1][0],
    reverse=True,
)[:3]

estimators = [(f"top{i+1}", clone(v[1])) for i, (_, v) in enumerate(top3)]
top3_display = [v[2] for _, v in top3]
print(f"Top-3 on '{best_feat}': {top3_display}")

voting = VotingClassifier(estimators=estimators, voting="soft")
_, _, metrics_v = run_experiment("voting", voting, {}, X_best, y, best_feat)
print(
    f"Voting ensemble  pr_auc={metrics_v['cv_pr_auc']:.4f}"
    f"  f1_tuned={metrics_v['cv_f1_tuned']:.4f}"
    f"  @t={metrics_v['cv_threshold']:.2f}"
)

## Best Run (MLflow Query)

Queries the MLflow tracking server directly — same logic used by `submit.py` — to confirm
which run will be selected for submission.

In [ ]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT)

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.cv_pr_auc DESC"],
)

desired_cols = [
    "run_id",
    "tags.model",
    "tags.feature_set",
    "params.optimal_threshold",
    "metrics.cv_pr_auc",
    "metrics.cv_auc",
    "metrics.cv_f1_tuned",
    "metrics.cv_threshold",
]
available_cols = [c for c in desired_cols if c in runs_df.columns]
display_df = runs_df[available_cols].head(10).copy()
display_df["run_id"] = display_df["run_id"].str[:8]

best = runs_df.iloc[0]
print(
    f"Best run: {best.run_id[:8]}"
    f"  model={best.get('tags.model', '?')}"
    f"  feat={best.get('tags.feature_set', '?')}"
    f"  cv_pr_auc={best.get('metrics.cv_pr_auc', 0):.4f}"
    f"  threshold={best.get('params.optimal_threshold', 0.5)}"
)
display_df